<a href="https://colab.research.google.com/github/anokhina-rgb/Google-Colabs/blob/main/Corpus_Processor_%26_Exporter_v4_db_vrt_tei.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# --- STEP 1: INSTALLATION & IMPORTS ---
!pip install simplemma feedparser beautifulsoup4 requests

import sqlite3
import pandas as pd
import simplemma
import json
import os
import re
import feedparser
import time
import requests
import shutil
from bs4 import BeautifulSoup
from datetime import datetime
from xml.etree.ElementTree import Element, SubElement, tostring
from xml.dom import minidom
from google.colab import files

# --- STEP 2: CONFIGURATION ---
LANGUAGES = ('uk', 'en', 'es', 'de')
DB_NAME = "daily_corpus.db"

RSS_FEEDS = {
    'ukraine_politics': 'https://www.ukrinform.ua/rss/politics',
    'world_news': 'http://feeds.bbci.co.uk/news/rss.xml',
    'europe_politics': 'https://euobserver.com/rss/politics',
    'spain_elpais_en': 'https://elpais.com/rss/elpais/inenglish.xml',
    'germany_dw': 'https://www.dw.com/xml/rss-de-all'
}

def prettify_xml(elem):
    """Форматування XML для зручного читання."""
    rough_string = tostring(elem, 'utf-8')
    reparsed = minidom.parseString(rough_string)
    return reparsed.toprettyxml(indent="  ")

def get_full_article_text(url):
    """Витягує повний текст статті."""
    try:
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}
        response = requests.get(url, headers=headers, timeout=15)
        if response.status_code == 200:
            soup = BeautifulSoup(response.content, 'html.parser')
            for s in soup(['script', 'style', 'nav', 'header', 'footer', 'aside', 'iframe', 'button']):
                s.decompose()
            selectors = ['article', 'div.story-fulltext', 'div[data-dtm-region="articulo_cuerpo"]', 'div.rich-text', 'main']
            for selector in selectors:
                found = soup.select_one(selector)
                if found:
                    paragraphs = found.find_all('p')
                    text = "\n\n".join([p.get_text().strip() for p in paragraphs if len(p.get_text().strip()) > 40])
                    if len(text) > 200: return text
    except: pass
    return None

def clean_html(html):
    return BeautifulSoup(html, "html.parser").get_text(separator=' ') if html else ""

def process_text(text, langs=LANGUAGES):
    """Токенізація та лематизація."""
    if not text: return []
    text = re.sub(r'\s+', ' ', str(text)).strip()
    tokens = simplemma.tokenizer.simple_tokenizer(text)
    return [{'token': t, 'lemma': simplemma.lemmatize(t, lang=langs)} for t in tokens]

# --- STEP 3: DATA COLLECTION ---

def init_db():
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS articles (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT, content TEXT, source TEXT, url TEXT,
            date_fetched TEXT, lang TEXT, category TEXT
        )
    ''')
    conn.commit()
    return conn

def fetch_daily_news(conn):
    print("🌐 Збір повних статей...")
    cursor = conn.cursor()
    new_count = 0
    for category, url in RSS_FEEDS.items():
        try:
            feed = feedparser.parse(url)
            for entry in feed.entries:
                cursor.execute("SELECT id FROM articles WHERE url = ?", (entry.link,))
                if not cursor.fetchone():
                    body = get_full_article_text(entry.link) or clean_html(entry.get('summary', ''))
                    if len(body) < 150: continue
                    lang = 'uk' if 'ukraine' in category else ('de' if 'germany' in category else 'en')
                    cursor.execute(
                        "INSERT INTO articles (title, content, source, url, date_fetched, lang, category) VALUES (?, ?, ?, ?, ?, ?, ?)",
                        (entry.title, body, feed.feed.get('title', category), entry.link, datetime.now().strftime("%Y-%m-%d"), lang, category)
                    )
                    new_count += 1
        except: continue
    conn.commit()
    return new_count

# --- STEP 4: EXPORT MULTIPLE FORMATS ---

def run_pipeline():
    conn = init_db()
    conn.row_factory = sqlite3.Row
    new_rss = fetch_daily_news(conn)
    print(f"✅ Додано {new_rss} нових статей.")

    cursor = conn.cursor()
    cursor.execute("SELECT * FROM articles")
    rows = cursor.fetchall()
    if not rows:
        print("❌ Немає даних для експорту."); return

    export_dir = "multi_format_corpus"
    if os.path.exists(export_dir): shutil.rmtree(export_dir)

    # Створюємо підпапки для форматів
    subdirs = ['plain', 'tagged', 'vrt', 'tei']
    for sd in subdirs: os.makedirs(os.path.join(export_dir, sd))

    print(f"📦 Експорт {len(rows)} статей у 4 форматах...")

    metadata_all = []

    for row in rows:
        safe_title = re.sub(r'[^\w\s-]', '', row['title']).strip().replace(' ', '_')[:50]
        base_name = f"{row['id']}_{safe_title}"
        annotations = process_text(row['content'])

        # 1. PLAIN (Сирий текст + Метадані)
        with open(os.path.join(export_dir, 'plain', f"{base_name}.txt"), 'w', encoding='utf-8') as f:
            f.write(f"TITLE: {row['title']}\nSOURCE: {row['source']}\nURL: {row['url']}\nDATE: {row['date_fetched']}\nLANG: {row['lang']}\n")
            f.write("-" * 30 + "\n\n" + row['content'])

        # 2. TAGGED (Токен/Лема)
        with open(os.path.join(export_dir, 'tagged', f"{base_name}_tagged.txt"), 'w', encoding='utf-8') as f:
            tagged_str = " ".join([f"{a['token']}/{a['lemma']}" for a in annotations])
            f.write(tagged_str)

        # 3. VRT (Verticalized для корпусного менеджеру)
        with open(os.path.join(export_dir, 'vrt', f"{base_name}.vrt"), 'w', encoding='utf-8') as f:
            f.write(f'<doc id="{row["id"]}" title="{row["title"]}" lang="{row["lang"]}">\n')
            for a in annotations: f.write(f"{a['token']}\t{a['lemma']}\n")
            f.write('</doc>')

        # 4. TEI XML (Архівний стандарт з метаданими)
        tei = Element('TEI', xmlns="http://www.tei-c.org/ns/1.0")
        header = SubElement(tei, 'teiHeader')
        file_desc = SubElement(header, 'fileDesc')
        title_stmt = SubElement(file_desc, 'titleStmt')
        t = SubElement(title_stmt, 'title'); t.text = row['title']
        pub_stmt = SubElement(file_desc, 'publicationStmt')
        p_date = SubElement(pub_stmt, 'date'); p_date.text = row['date_fetched']
        text_tag = SubElement(tei, 'text')
        body = SubElement(text_tag, 'body')
        p = SubElement(body, 'p')
        for a in annotations:
            w = SubElement(p, 'w', lemma=a['lemma']); w.text = a['token']

        with open(os.path.join(export_dir, 'tei', f"{base_name}.xml"), 'w', encoding='utf-8') as f:
            f.write(prettify_xml(tei))

        # Додаємо в загальний індекс
        metadata_all.append(dict(row))

    # Зберігаємо загальний JSON
    with open(os.path.join(export_dir, 'corpus_metadata.json'), 'w', encoding='utf-8') as f:
        json.dump(metadata_all, f, ensure_ascii=False, indent=2)

    # Архівація
    shutil.make_archive("full_corpus_bundle", 'zip', export_dir)
    conn.close()

    print("\n🚀 Готово! Завантажуємо ZIP-архів та файл бази даних SQLite.")

    # Почергове завантаження
    files.download("full_corpus_bundle.zip")
    time.sleep(1.5)
    if os.path.exists(DB_NAME):
        files.download(DB_NAME)

if __name__ == "__main__":
    run_pipeline()

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.2/67.2 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 2.2 MB/s eta 0:00:00
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6046 sha256=0d7b4f4356eda8b2548504d83015beb5321bd4b2a1ae241466a9028b5f966b4c
  Stored in directory: /root/.cache/pip/wheels/03/f5/1a/23761066dac1d0e8e683e5fdb27e12de53209d05a4a37e6246
Successfully built sgmllib3k
🌐 Збір повних статей...
✅ Додано 92 нових статей.
📦 Експорт 92 статей у 4 форматах...

🚀 Готово! Завантажуємо ZIP-архів та файл бази даних SQLite.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>